<a href="https://colab.research.google.com/github/Changin7/ALURA--Challenge-telecom-X/blob/main/TelecomX_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Extracción de Datos (Enlace Original Mantenido)
url_datos = 'https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json'
respuesta = requests.get(url_datos)
datos_json = respuesta.json()

# Normalización del JSON a DataFrame
df_telecom = pd.json_normalize(datos_json, sep='_')

# 2. Limpieza y Transformación de Datos
print(f"Filas duplicadas antes de limpiar: {df_telecom.duplicated().sum()}")
df_telecom = df_telecom.drop_duplicates()

# Reemplazar espacios vacíos por NaN y forzar tipo numérico en los cargos totales
df_telecom = df_telecom.replace(' ', np.nan)
df_telecom['account_Charges_Total'] = df_telecom['account_Charges_Total'].astype(float)

# Limpiar espacios al inicio/final de las variables de texto y homogeneizar nulos
columnas_texto = df_telecom.select_dtypes(include=['object']).columns
for col in columnas_texto:
    df_telecom[col] = df_telecom[col].str.strip().replace('', np.nan)

# Descartar filas con datos faltantes
df_telecom = df_telecom.dropna()

# Poner todos los nombres de las columnas en minúsculas
df_telecom.columns = df_telecom.columns.str.lower()

# Ingeniería de variables: Crear columna de 'cargos_diarios'
df_telecom['account_charges_day'] = (df_telecom['account_charges_monthly'] / 30).fillna(0)

# Estandarización: Convertir respuestas afirmativas/negativas a binarios (1 y 0)
diccionario_mapeo = {
    'yes': 1, 'no': 0, 'Yes': 1, 'No': 0,
    'Male': 1, 'Female': 0,
    'No internet service': 0, 'No phone service': 0
}

for col in df_telecom.select_dtypes(include=['object']).columns:
    df_telecom[col] = df_telecom[col].replace(diccionario_mapeo).infer_objects(copy=False)

# Asegurar que las columnas booleanas sean int64
for col in df_telecom.select_dtypes(include=['bool']).columns:
    df_telecom[col] = df_telecom[col].astype(int)

# Verificación final
df_telecom.info()

In [ ]:
# Mostrar estadísticas descriptivas formateadas
resumen_stats = df_telecom.describe().style.format('{:.2f}')
resumen_stats.set_sticky(axis='index')
display(resumen_stats) # Usa 'display' para que en Colab se vea como una tabla bonita

# Gráfico Circular: Proporción de Abandono (Churn)
plt.figure(figsize=(7, 7))
colores_pie = ['#66b3ff', '#ff9999']
df_telecom['churn'].value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',
    colors=colores_pie,
    startangle=90,
    explode=(0, 0.05) # Pequeña separación para destacar el abandono
)
plt.title('Porcentaje de Evasión (Churn) de Clientes', fontweight='bold')
plt.legend(labels=['Activos', 'Baja (Evasión)'])
plt.ylabel('')
plt.show()

# Proporción del servicio de Internet
print("\n--- Distribución de Internet ---")
print(df_telecom['internet_internetservice'].value_counts(normalize=True) * 100)

# Mapa de Calor de Correlaciones (Heatmap)
df_numerico = df_telecom.select_dtypes(include=[np.number])
matriz_corr = df_numerico.corr()

# Máscara para que se vea solo el triángulo inferior (diseño más limpio)
mascara = np.triu(np.ones_like(matriz_corr, dtype=bool))

plt.figure(figsize=(14, 10))
sns.heatmap(matriz_corr, mask=mascara, annot=True, cmap='magma', fmt='.2f', vmin=-1, vmax=1)
plt.title('Mapa de Correlación de Variables', fontsize=16, pad=20, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import shapiro, mannwhitneyu, chi2_contingency

# Separar DataFrames por estatus del cliente
clientes_activos = df_telecom.query('churn == 0')
clientes_baja = df_telecom.query('churn == 1')

# ---- 1. DISTRIBUCIÓN Y NORMALIDAD (Cargos Mensuales) ----
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))
sns.kdeplot(data=clientes_activos['account_charges_monthly'], ax=ejes[0], fill=True, color='blue')
ejes[0].set_title('Cargos Mensuales - Clientes Activos')

sns.kdeplot(data=clientes_baja['account_charges_monthly'], ax=ejes[1], fill=True, color='red')
ejes[1].set_title('Cargos Mensuales - Clientes Baja')
plt.show()

# Test de Shapiro-Wilk
est_act, p_act = shapiro(clientes_activos['account_charges_monthly'])
est_baj, p_baj = shapiro(clientes_baja['account_charges_monthly'])
print("Test de Normalidad Shapiro-Wilk (Cargos Mensuales):")
print(f"Activos -> P-value: {p_act} | Bajas -> P-value: {p_baj}\n")

# ---- 2. TEST MANN-WHITNEY U (Diferencia de medianas) ----
# ¿Pagan más los que abandonan? (alternative='less')
u_stat_cargos, p_val_cargos = mannwhitneyu(
    clientes_activos['account_charges_monthly'],
    clientes_baja['account_charges_monthly'],
    alternative='less'
)
print("Test Mann-Whitney U (Cargos Mensuales):")
print(f"Estadístico U: {u_stat_cargos} | P-valor: {p_val_cargos}")

# ¿Tienen mayor antigüedad los activos comparados con los que abandonan? (alternative='greater')
u_stat_antig, p_val_antig = mannwhitneyu(
    clientes_activos['customer_tenure'],
    clientes_baja['customer_tenure'],
    alternative='greater'
)
print("\nTest Mann-Whitney U (Antigüedad):")
print(f"Estadístico U: {u_stat_antig} | P-valor: {p_val_antig}\n")

# ---- 3. TEST DE CHI-CUADRADO (Contrato vs Evasión) ----
tabla_contingencia = pd.crosstab(df_telecom['account_contract'], df_telecom['churn'])
chi2_stat, p_val_chi2, grados_lib, valores_esp = chi2_contingency(tabla_contingencia)

print("Test Chi-Cuadrado (Tipo de Contrato vs Churn):")
print(f"Chi-cuadrado: {chi2_stat:.4f} | P-valor: {p_val_chi2}")

if p_val_chi2 < 0.05:
    print(">> CONCLUSIÓN: Se RECHAZA H0. El tipo de contrato SÍ está significativamente relacionado con la evasión de clientes.")
else:
    print(">> CONCLUSIÓN: NO se rechaza H0. No hay relación significativa.")

# Gráfico de barras de proporciones para Chi-Cuadrado
tabla_porcentajes = pd.crosstab(df_telecom['account_contract'], df_telecom['churn'], normalize='index') * 100
tabla_porcentajes.plot(kind='bar', figsize=(10, 6), color=['#5cb85c', '#d9534f'])
plt.title('Proporción de Evasión por Tipo de Contrato', fontweight='bold')
plt.xlabel('Tipo de Contrato')
plt.ylabel('Porcentaje (%)')
plt.legend(['Permanecen', 'Abandonan'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()